In [14]:
import requests
import os
from openai import OpenAI

# REEMPLAZO OBLIGATORIO PARA GITHUB (Cámbialo al final)
API_TOKEN = "TU_TOQUEN"

# Inicializar el cliente de OpenAI para el Hugging Face Router
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=API_TOKEN,
)

def llamar_modelo(prompt, temperatura=0.7, model="google/gemma-7b-it"):
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=temperatura,
            max_tokens=500,
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:

        print(f"Error de la API: {e}")
        return f"Error de conexión: {str(e)}"

In [15]:
# Define la personalidad del chatbot
SYSTEM_PROMPT = """
Eres Pitón, un experto tutor de Python para principiantes. Respondes siempre en español, de forma amigable y didáctica. Solo puedes responder preguntas relacionadas con la programación en Python y conceptos básicos de informática que ayuden a entender Python. Si te hacen una pregunta fuera de tu tema, o si la pregunta es muy avanzada para un principiante, amablemente dirás que no estás preparado para responder esa pregunta. Siempre proporciona ejemplos de código claros y sencillos cuando sea apropiado.
"""

print("System prompt definido:")
print("-" * 40)
print(SYSTEM_PROMPT)

System prompt definido:
----------------------------------------

Eres Pitón, un experto tutor de Python para principiantes. Respondes siempre en español, de forma amigable y didáctica. Solo puedes responder preguntas relacionadas con la programación en Python y conceptos básicos de informática que ayuden a entender Python. Si te hacen una pregunta fuera de tu tema, o si la pregunta es muy avanzada para un principiante, amablemente dirás que no estás preparado para responder esa pregunta. Siempre proporciona ejemplos de código claros y sencillos cuando sea apropiado.



In [16]:
def chatbot_responder(pregunta_usuario, temperatura=0.7):
    """Genera una respuesta usando el system prompt y la pregunta."""
    # Formato de prompt para Zephyr (instrucción + mensaje)
    prompt_completo = f"""<|system|>
{SYSTEM_PROMPT}
<|user|>
{pregunta_usuario}
<|assistant|>"""
    respuesta = llamar_modelo(prompt_completo, temperatura)
    return respuesta

# Prueba con una primera pregunta
mi_pregunta = "Hola, ¿quién eres y en qué me puedes ayudar?"
resp = chatbot_responder(mi_pregunta)
print(f"Usuario: {mi_pregunta}")
print(f"Chatbot: {resp}")

Error de la API: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}}
Usuario: Hola, ¿quién eres y en qué me puedes ayudar?
Chatbot: Error de conexión: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}}


## ── MILESTONE 3: Historial de Conversación ───────────────

Ahora implementamos un chatbot que recuerda el contexto de la conversación, agregando los mensajes del usuario y las respuestas del asistente al historial.

In [17]:
historial = [] # lista de {role, content}

def chatbot_con_memoria(pregunta_usuario, temperatura=0.7):
    """Chatbot que recuerda el historial de la conversación."""
    # Agregar mensaje del usuario al historial
    historial.append({"role": "user", "content": pregunta_usuario})

    # Construir el prompt con todo el historial
    prompt = f"<|system|>\n{SYSTEM_PROMPT}\n"
    for msg in historial:
        rol = "user" if msg["role"] == "user" else "assistant"
        prompt += f"<|{rol}|>\n{msg['content']}\n"
    prompt += "<|assistant|>"

    # Llamar al modelo
    respuesta = llamar_modelo(prompt, temperatura)

    # Agregar respuesta al historial
    historial.append({"role": "assistant", "content": respuesta})
    return respuesta

def reiniciar_chat():
    """Limpia el historial para empezar una nueva conversación."""
    global historial
    historial = []
    print("✓ Conversación reiniciada")

# Prueba: conversación de 3 turnos
reiniciar_chat()
for pregunta in [
    "Hola, ¿puedes ayudarme?",
    "¿Qué fue lo primero que te pregunté?", # prueba de memoria
    "Gracias, ¿puedes resumir nuestra conversación?"
]:
    print(f"\n👤 {pregunta}")
    print(f"🤖 {chatbot_con_memoria(pregunta)}")

✓ Conversación reiniciada

👤 Hola, ¿puedes ayudarme?
Error de la API: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}}
🤖 Error de conexión: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}}

👤 ¿Qué fue lo primero que te pregunté?
Error de la API: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}}
🤖 Error de conexión: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 

## ── MILESTONE 4: Loop de Conversación Interactivo ────────

Este es el bucle principal donde puedes interactuar con el chatbot en tiempo real. Escribe tus preguntas y el chatbot responderá, manteniendo el contexto de la conversación.

In [ ]:
# IMPORTANTE: Ejecuta esta celda y escribe en el campo de input
# Escribe "salir" para terminar la conversación
reiniciar_chat()
print("═══════════════════════════════════════")
print(f" Chatbot activado — tema: {SYSTEM_PROMPT[:50]}...")
print(" Escribe 'salir' para terminar")
print("═══════════════════════════════════════\n")

while True:
    entrada = input("👤 Tú: ").strip()
    if entrada.lower() in ["salir", "exit", "quit", "bye"]:
        print("🤖 Hasta luego. ¡Fue un placer ayudarte!")
        break
    if not entrada:
        continue
    respuesta = chatbot_con_memoria(entrada)
    print(f"🤖 Chatbot: {respuesta}\n")

✓ Conversación reiniciada
═══════════════════════════════════════
 Chatbot activado — tema: 
Eres Pitón, un experto tutor de Python para princ...
 Escribe 'salir' para terminar
═══════════════════════════════════════

Error de la API: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}}
🤖 Chatbot: Error de conexión: Error code: 400 - {'error': {'message': "The requested model 'google/gemma-7b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}}



## ── MILESTONE 5: Mejoras de Calidad (Opcional) ─────────

Estas funciones opcionales mejoran el chatbot con un límite en el historial de conversación para evitar prompts muy largos y una forma de ver estadísticas de la conversación.

In [ ]:
# 1. Límite de historial (evitar prompts muy largos)
def chatbot_mejorado(pregunta, temperatura=0.7, max_historial=6):
    """Chatbot con historial limitado y estadísticas."""
    historial.append({"role": "user", "content": pregunta})

    # Limitar a los últimos N mensajes para no saturar el contexto
    hist_reciente = historial[-max_historial:]

    prompt = f"<|system|>\n{SYSTEM_PROMPT}\n"
    for msg in hist_reciente:
        rol = "user" if msg["role"] == "user" else "assistant"
        prompt += f"<|{rol}|>\n{msg['content']}\n"
    prompt += "<|assistant|>"

    respuesta = llamar_modelo(prompt, temperatura)
    historial.append({"role": "assistant", "content": respuesta})
    return respuesta

# 2. Mostrar estadísticas de la conversación
def stats_conversacion():
    """Muestra estadísticas del chat actual."""
    turnos = len([m for m in historial if m["role"]=="user"])
    palabras = sum(len(m["content"].split()) for m in historial)
    print(f"Turnos de conversación: {turnos}")
    print(f"Palabras totales: {palabras}")
    print(f"Mensajes en historial: {len(historial)}")
